In [122]:
import kenlm
from transformers import AutoTokenizer
import numpy as np
from unigramlm import UnigramLM

In [124]:
unigram_lm = UnigramLM("../models/unigrams/babylm.csv")
unigram_lm.load_counts()

In [3]:
lm = kenlm.Model("../models/fourgrams/babylm.txt.binary")
try:
    tokenizer = AutoTokenizer.from_pretrained(
        f"kanishka/smolm-autoreg-bpe-babylm-1e-3"
    )
except:
    tokenizer = AutoTokenizer.from_pretrained(
        f"kanishka/smolm-autoreg-bpe-babylm-3e-4"
    )

tokenizer_config.json:   0%|          | 0.00/475 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/249k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/145k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/672k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

In [99]:
tokenized = tokenizer.tokenize(" " + "The family spent an astonishing three weeks")

In [100]:
tokenized

['Ġthe', 'Ġfamily', 'Ġspent', 'Ġan', 'Ġaston', 'ishing', 'Ġthree', 'Ġweeks']

In [101]:
scores = [p[0] for p in list(lm.full_scores(" ".join(tokenized)))[:-1]]

In [102]:
np.sum(scores) / len(tokenized)

-2.2299264492467046

In [44]:
list(lm.full_scores(' '.join(tokenizer.tokenize("The"))))

[(-1.3637523651123047, 2, False), (-2.174959182739258, 3, False)]

In [144]:
def four_gram(prefix, continuation):
    unigram_logprob = unigram_lm.sentence_log_prob(" " + continuation)

    full = f"{prefix} {continuation}"
    full_tokenized = tokenizer.tokenize(" " + full)
    # print(full_tokenized)
    full_length = len(full_tokenized)

    scores = list(lm.full_scores(" ".join(full_tokenized)))

    prefix_tokenized = tokenizer.tokenize(" " + prefix)
    # print(prefix_tokenized)
    prefix_len = len(prefix_tokenized)

    region = [p[0] for p in scores][prefix_len:-1]

    return (np.sum(region) / (full_length - prefix_len)) - unigram_logprob

In [ ]:
four_gram("The family spent", "a beautiful three weeks")

4.3622866313011945

In [120]:
list(lm.full_scores(" ".join(['Ġthe', 'Ġfamily', 'Ġspent', 'Ġan', 'Ġaston', 'ishing', 'Ġthree', 'Ġweeks'])))

[(-1.3637523651123047, 2, False),
 (-2.850321054458618, 3, False),
 (-2.4238321781158447, 4, False),
 (-2.625626564025879, 2, False),
 (-3.570128917694092, 2, False),
 (-0.08918390423059464, 3, False),
 (-2.5853962898254395, 4, False),
 (-2.3311703205108643, 2, False),
 (-1.5238465070724487, 3, False)]

In [121]:
list(lm.full_scores(" ".join(['Ġthe', 'Ġfamily', 'Ġspent', 'Ġa', 'Ġthree', 'Ġaston', 'ishing', 'Ġweeks'])))

[(-1.3637523651123047, 2, False),
 (-2.850321054458618, 3, False),
 (-2.4238321781158447, 4, False),
 (-1.120948076248169, 4, False),
 (-3.750635862350464, 2, False),
 (-5.791816711425781, 1, False),
 (-0.693530261516571, 2, False),
 (-4.787260055541992, 1, False),
 (-1.808652639389038, 2, False)]

In [128]:
unigram_lm.sentence_log_prob("an three")

-6.830191651107796